# Historical second-prior experiment

This is a retrospective note, not a maintained training recipe. Around commit `44f8854` (`feat: implement second diffusion`, 2025-11-01), the prior was extended to model a second image-latent target alongside the primary CLIP image latent. The approach was later removed in commit `2cca62a` (`removed second prior`, 2025-12-07) after the recorded experiments did not establish a reliable working improvement.

## What changed at `44f8854`

- Added `prior_img_encoder_2`, initially `unaligned_synclr_vitb16`, and a `prior_align_second_latent` flag.
- Loaded statistics and cached embeddings for both `prior_img_latent` and `prior_img_latent_2`.
- Concatenated the two target latents, increasing the prior input dimension to the sum of both dimensions.
- Added a second-latent alignment loss in addition to the primary prior losses.
- Added second-latent validation metrics such as `prior/pred2_cos` and `prior/pred2_align_top1`.

The later `train_eeg_prior_align.yaml` configuration experimented with this family of ideas, including a conditioning variant. Commit `2cca62a` removed the second-prior fields, data loading, losses, metrics, and configs.

In [1]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RUN_ROOT = PROJECT_ROOT / 'experiments/prior_second'
rows = []
for metrics_path in sorted(RUN_ROOT.glob('*/version_0/test_metrics.json')):
    values = json.loads(metrics_path.read_text())
    row = {'run': metrics_path.parent.parent.name}
    row.update({key.removeprefix('prior/'): value for key, value in values.items()})
    rows.append(row)
results = pd.DataFrame(rows)
results.round(4)

,run,pixcorr,ssim,alex2,alex5,inceptionv3,clip,efficientnet,swav,pred2_align_top1,pred2_cos
0,251111230700bahqib-slurm14700183,0.0778,0.3112,0.5354,0.5392,0.5609,0.5580,0.9586,0.6603,0.015,0.7190
1,251111230700tovphb-slurm14700182,0.1554,0.3094,0.7632,0.8084,0.6554,0.7295,0.9182,0.6128,0.155,0.8004
2,251111230720iokhhq-slurm14700184,0.0921,0.3041,0.6709,0.7547,0.6320,0.7455,0.9335,0.6133,NaN,NaN


## Recorded outcome

The archived runs were inconsistent. The strongest recorded second-latent retrieval result was `pred2_align_top1 = 0.155` with `pred2_cos = 0.800`; another run reached `pred2_cos = 0.719` and `pred2_align_top1 = 0.015`. Reconstruction metrics also varied substantially, and there was no stable evidence that the second prior improved the primary reconstruction objective. These results are why the second prior remains historical rather than part of the current prior-only baseline.

In [2]:
columns = ['run', 'pixcorr', 'ssim', 'alex2', 'alex5', 'inceptionv3', 'clip', 'swav', 'pred2_align_top1', 'pred2_cos']
available = [column for column in columns if column in results.columns]
results[available].sort_values('pred2_cos', ascending=False, na_position='last').round(4)

,run,pixcorr,ssim,alex2,alex5,inceptionv3,clip,swav,pred2_align_top1,pred2_cos
1,251111230700tovphb-slurm14700182,0.1554,0.3094,0.7632,0.8084,0.6554,0.7295,0.6128,0.155,0.8004
0,251111230700bahqib-slurm14700183,0.0778,0.3112,0.5354,0.5392,0.5609,0.5580,0.6603,0.015,0.7190
2,251111230720iokhhq-slurm14700184,0.0921,0.3041,0.6709,0.7547,0.6320,0.7455,0.6133,NaN,NaN


## Current status

The maintained workflow intentionally uses one prior target and one reconstruction path. Reintroducing a second prior would require a new controlled experiment with an explicit target, conditioning/concatenation design, loss weighting, checkpoint metric, and apples-to-apples reconstruction evaluation.